# 08 — Batch Processing & Document Comparison

Process multiple documents in parallel with `process_batch()` and compare extracted fields across them with `compare_documents()`.

## Processing documents at scale

Extracting data from a single document is useful; extracting from hundreds or thousands is where real value emerges. But batch processing is not just "run the same pipeline in a loop" — it introduces aggregate signals that individual extractions cannot provide.

### What batch reports tell you

A `BatchReport` gives you fleet-level visibility into your extraction pipeline:

- **Success/failure rates** — what percentage of documents processed without errors? Failures often cluster around specific document types or formats.
- **Average confidence** — is your pipeline consistently confident, or are there outliers dragging the average down?
- **Top review reasons** — if the same review rule fires on 40% of documents, that is a systematic issue worth investigating (too-tight threshold? bad schema description? parser mismatch?).
- **Per-document summaries** — which specific documents failed or need review, and why.

The CSV and DataFrame exports turn these into data you can feed into dashboards, alerting systems, or further analysis.

### Document comparison for correctness

`compare_documents()` extracts from multiple files in parallel and then does a **field-by-field diff** across all results. This is useful in two scenarios:

1. **Version comparison** — you have multiple versions of the same document (draft, final, signed) and want to know what changed. Fields where all versions agree are stable; fields that differ are worth investigating.

2. **Cross-validation** — you have the same document processed by different pipelines (or at different times) and want to catch extraction errors. If three independent extractions all agree on a value, it is almost certainly correct. If they disagree, at least one is wrong.

The comparison produces a `FieldDifference` per field with `all_agree`, `unique_values`, and `value_counts` — everything you need to decide whether a disagreement matters.

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

FILES = [
    "data/sample_invoice.pdf",
    "data/sample_invoice_2.pdf",
    "data/sample_invoice_3.pdf",
]

from pydantic import BaseModel, Field


class Invoice(BaseModel):
    supplier_name: str = Field(description="Supplier company name")
    invoice_number: str = Field(description="Invoice reference number")
    invoice_date: str = Field(description="Date invoice was issued")
    due_date: str = Field(description="Payment due date")
    po_number: str = Field(description="Purchase order number")
    total: float = Field(description="Grand total including tax")
    subtotal: float = Field(description="Subtotal before tax")
    tax_amount: float = Field(description="Tax amount")
    payment_terms: str = Field(description="Payment terms")
    bill_to_company: str = Field(description="Billing company name")

## Batch Processing

Extract all three invoices in one call. Each document is processed concurrently.

In [ ]:
from docflow import process_batch, DocumentPipeline

pipeline = DocumentPipeline(parser="pymupdf", model="gemini/gemini-2.5-flash")
report = process_batch(files=FILES, schema=Invoice, pipeline=pipeline)

## Batch Report

In [ ]:
print(f"Total:            {report.total}")
print(f"Succeeded:        {report.succeeded}")
print(f"Failed:           {report.failed}")
print(f"Needs review:     {report.needs_review}")
print(f"Avg confidence:   {report.average_confidence:.2f}")
if report.top_review_reasons:
    print(f"\nTop review reasons:")
    for reason, count in report.top_review_reasons.items():
        print(f"  ({count}x) {reason}")

## Per-document Summary

In [ ]:
for doc in report.documents:
    status = "OK" if doc.success and not doc.needs_review else ("REVIEW" if doc.needs_review else "FAIL")
    conf = f"{doc.confidence:.2f}" if doc.success else "N/A"
    print(f"[{status:6}] {doc.file_name:<28} conf={conf}")
    if doc.success:
        print(f"         supplier={doc.data.get('supplier_name', 'N/A')}")
        print(f"         total={doc.data.get('total', 'N/A')}")
        print(f"         invoice_number={doc.data.get('invoice_number', 'N/A')}")

## CSV Export

In [ ]:
csv_output = report.to_csv()
print("CSV output (first 5 lines):")
for line in csv_output.split("\n")[:5]:
    print(f"  {line}")

## DataFrame Export

In [ ]:
try:
    df = report.to_dataframe()
    print(f"DataFrame shape: {df.shape}")
    print(df[["file_name", "success", "confidence", "supplier_name", "total"]].to_string())
except ImportError:
    print("pandas not installed — install with: pip install pandas")

## Document Comparison

Compare extracted fields across all three invoices to spot agreements and differences.

In [ ]:
from docflow import compare_documents

comparison = compare_documents(files=FILES, schema=Invoice, pipeline=pipeline)
print(f"Compared {len(comparison.documents)} documents")

## Field-by-field Differences

In [ ]:
print(f"\nField differences:")
for fname, diff in comparison.differences.items():
    icon = "SAME" if diff.all_agree else "DIFF"
    print(f"  [{icon:4}] {fname:<22} {diff.summary}")

## Deep Dive on Differences

For fields where documents disagree, show the unique values and how many documents report each.

In [ ]:
for fname, diff in comparison.differences.items():
    if not diff.all_agree:
        print(f"\n{fname}:")
        print(f"  Unique values: {diff.unique_values}")
        if diff.value_counts:
            for val, count in diff.value_counts.items():
                print(f"    {val}: {count}/{len(comparison.documents)} documents")

## Quality Across the Batch

In [ ]:
from docflow import quality_report

batch_quality = quality_report(report.results)
print(f"Batch quality score: {batch_quality.score:.4f}")
print(f"Worst fields: {batch_quality.worst_fields}")